## Zelf een NN maken zonder library

We gaan nu zelf een Neuraal NEtwerk implementeren zonder gebruik te maken van libraries voor Neurale Netwerken. Sklearn gebruiken om de mnsit data te laden en numpy gebruiken mag uiteraard wel.

### Data laden

Laad de mnist data zoals je het eerder hebt geladen. Split dedara op in een test/training set. Bepaal zelf op hoeveel je split.

Normaliseer de data op 0-1 ipv de pixelwaardes. Dit heb je ook al eerder gedaan. 

Doe een one hot encoding op de labels van de data.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh  = encoder.transform(y_test.reshape(-1, 1))

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train_oh: {y_train_oh.shape}")

### Activatie functies

Maak de volgende activatiefuncties: 

    def relu(waardes)

en 

    def softmax(waardes) 

Als het goed is kun je ze uit de vorige opdracht gebruiken. 

In [ ]:
def relu(z):
    return np.maximum(0, z)

def softmax(z):
    e = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

### Cross entropy loss

Maak een functie

    def cross_entropy(y_true, y_predicted)

functie om de grootte van loss te kunnen berekenen. We gaan de waarde van de loss gebruiken als validatie output. Het wordt dus aan het einde geprint.

In [ ]:
def cross_entropy(y_true, y_predicted):
    return -np.mean(np.sum(y_true * np.log(y_predicted + 1e-8), axis=1))

### Initialiseer je NN 

Maak de volgende variabelen:

- Kies hoeveel input nodes je hebt. 
- En hoeveel hidden layer nodes? 
- En hoeveel output nodes? Leg uit waarom je dat hebt gekozen.


Maak voor nu 1 input layer en 1 hidden layer. JE mag dit later uitbreiden

Om de initiele gewichten een random waarde te geven kun je np.random.randn gebruiken. Deze geeft een normale verdeling met waarden die ongeveer liggen tussen -3 en +3.

Om grote getallen te voorkomen kun je deze weer kleiner maken door bijv *0.01 te doen, dan wordt de standaardafwijking 0.01

In [ ]:
# Input: 784 nodes (28x28 pixels, elk als aparte input)
# Hidden: 64 nodes (genoeg capaciteit zonder te zwaar te zijn)
# Output: 10 nodes (één per cijfer 0-9)
input_nodes  = 784
hidden_nodes = 64
output_nodes = 10

W1 = np.random.randn(input_nodes, hidden_nodes) * 0.01
b1 = np.zeros(hidden_nodes)
W2 = np.random.randn(hidden_nodes, output_nodes) * 0.01
b2 = np.zeros(output_nodes)

print(f"W1: {W1.shape}, b1: {b1.shape}")
print(f"W2: {W2.shape}, b2: {b2.shape}")

### Forward pass
Maak een forward propagation functie. We returnen ook tussenwaarden (cache) zodat we die voor backpropagation runnen gebruiken.

#### Stap 1

Maak functie def forward()
Als parameters hgebruik de volgende:
- inputdata X
- gewichtsmatrix W1 van de hidden layer nodes
- biasvector b1 van de hidden layer nodes
- gewichtsmatrix W2 van de output layer nodes
- biasvector b2 van de output layer nodes

Bereken eerst de invoer van de hidden layer. Vermenigvuldig de input met de gewichten en tel de bias erbij op.


Hint:

denk aan matrixvermenigvuldiging
de vorm moet kloppen: (samples, features) @ (features, hidden)

#### Stap 2

De uitkomst van stap 1 is de ruwe activatie van de hidden layer. Pas nu de door jou geiplementeerde ReLU-functie hierop toe.

#### Stap 3

Gebruik de output van de hidden layer als input voor de outputlaag. Vermenigvuldig met W2 en tel de bias b2 er bij op. Eigenlijk bijna hetzelfde als wat je in stap 1 hebt gedaan maar met andere dimensies.

#### Stap 4

De output van stap 3 zijn “scores” (logits), nog geen kansen. Zet deze scores om naar kansen met de softmax-functie die je eerder hebt gemaakt.

#### Stap 5


Voor de backpropagation stap heb je straks de tussenwaarden nodig.
Sla alle relevante variabelen op in één object (bijv. tuple)

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

#### Stap 6

return nu (bijvoorbeeld in een tuple vorm) de voorspelling en de object uit Stap 5

In [ ]:
def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1        # stap 1: pre-activatie hidden layer
    A1 = relu(Z1)            # stap 2: activatie hidden layer
    Z2 = A1 @ W2 + b2       # stap 3: pre-activatie output layer
    A2 = softmax(Z2)         # stap 4: output kansen
    cache = (X, Z1, A1, Z2, A2)  # stap 5: tussenwaarden opslaan
    return A2, cache         # stap 6: return

### Backward pass

We gaan enkele functies voorbereiden die we in de les hebben gehad om de backpropagation stap uit te voeren.

We beginnen met berekening van de gradient van loss.

Tip: Je kunt de formule uit de slides van deze week gebruiken. 

Maak nu een methode 
        
        def compute_output_gradient(final_prediction, correct_answers)
        
die de output (laag) gradient teruggeeft. Je kunt de formule hierboven gebruiken.


In [ ]:
def compute_output_gradient(final_prediction, correct_answers):
    return final_prediction - correct_answers

Maak nu methode 


    def compute_output_gradients(hidden_output, output_gradient)

die de gradienten van de output‑gewichten teruggeeft. Voeg aan de output van je methode ook de gradient van bias toe (als tuple bijv.). 

Tip: Je kunt de formule uit de slides van deze week gebruiken.


In [ ]:
def compute_output_gradients(hidden_output, output_gradient):
    n = hidden_output.shape[0]
    dW2 = hidden_output.T @ output_gradient / n
    db2 = np.mean(output_gradient, axis=0)
    return dW2, db2

Maak de methode 

    def compute_hidden_gradient(output_gradient, hidden_to_output_weights)
    
Deze zegt hoe sterk elke hidden neuron bijdraagt aan de fout.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

In [ ]:
def compute_hidden_gradient(output_gradient, hidden_to_output_weights):
    return output_gradient @ hidden_to_output_weights.T

Schrijf een python functie die afgeleide RELU berekent voor een getal:

    def relu_derivative(x)

Pas het aan zodat het voor een lijst van getallen werkt (eg met numpy)


In [ ]:
def relu_derivative(x):
    return (x > 0).astype(float)

Maak nu methode 

    def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data)

De functie moet de gradienten van de hidden-laag gewichten en bias teruggeven. Je hebt de afgeleide van de RELU nodig uit de vorige opgave, want alleen die gewichten doen mee.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

In [ ]:
def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data):
    n = input_data.shape[0]
    dZ1 = hidden_gradient * relu_derivative(hidden_raw_gradient)
    dW1 = input_data.T @ dZ1 / n
    db1 = np.mean(dZ1, axis=0)
    return dW1, db1

Maak nu de backward propagation nmethode die de bovenstaatde methodes zal gbruiken.

    def backward(y_true, cache, W2):

De parameter cache zal uit de forward pass komen en als het goed is de volgende waardes bevatten:

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

Die kun je dus daaruit unpacken.

Doe nu de volgende stappen achter elkaar:

- Bereken de output gradient (gebruik je eerder gemaakte methode)
- Bereken de output gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en output gradient)

- Bereken de hidden gradient (gebruik je eerder gemaakte methode)
- Bereken de hidden gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en hidden gradient)

return de output gradients en de hidden gradients bijvoorbeeld in de vorm van een tuple:

    return dW1, db1, dW2, db2

In [ ]:
def backward(y_true, cache, W2):
    X, Z1, A1, Z2, A2 = cache

    dZ2       = compute_output_gradient(A2, y_true)
    dW2, db2  = compute_output_gradients(A1, dZ2)

    dA1       = compute_hidden_gradient(dZ2, W2)
    dW1, db1  = compute_hidden_gradients(dA1, Z1, X)

    return dW1, db1, dW2, db2

### De training loop. 

JE mag het volgende stukje code gebruiken om je netwerk te trainen. MAar je mag uiteraard ook zelf een andere opzet maken.


In [ ]:
# bedenk een learning rate die je wilt gebruiken
lr = 0.1

for epoch in range(20):

    # forward pass
    voorspelling, cache = forward(X_train, W1, b1, W2, b2)
    
    # loss berekenen we wel, maar doen er verder niet echt wat mee behalve printen
    # y_train_oh staat voor voor one hot encoded
    loss = cross_entropy(y_train_oh, voorspelling)

    # backward pass
    dW1, db1, dW2, db2 = backward(y_train_oh, cache, W2)

    # update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 5 == 0:
        preds = np.argmax(voorspelling, axis=1)

        # voor accuracy gebruik je class labels, niet one-hot vectors
        acc = np.mean(preds == y_train)
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Acc: {acc:.3f}")

Epoch 0, Loss: 0.4247, Acc: 0.895
Epoch 5, Loss: 0.4194, Acc: 0.896
Epoch 10, Loss: 0.4144, Acc: 0.897
Epoch 15, Loss: 0.4095, Acc: 0.898


One hot encoding toepassen (op de labels)

### Experimenteer

Werkt het? Mooi, dan kun je nu zelf experimenteren met verschillende setup parameters en kijken wat het beste werkt.

In [ ]:
# Experiment: meer hidden nodes (128) en hogere learning rate
W1_exp = np.random.randn(input_nodes, 128) * 0.01
b1_exp = np.zeros(128)
W2_exp = np.random.randn(128, output_nodes) * 0.01
b2_exp = np.zeros(output_nodes)

lr_exp = 0.5
for epoch in range(20):
    pred_exp, cache_exp = forward(X_train, W1_exp, b1_exp, W2_exp, b2_exp)
    loss_exp = cross_entropy(y_train_oh, pred_exp)
    dW1, db1, dW2, db2 = backward(y_train_oh, cache_exp, W2_exp)
    W1_exp -= lr_exp * dW1
    b1_exp -= lr_exp * db1
    W2_exp -= lr_exp * dW2
    b2_exp -= lr_exp * db2
    if epoch % 5 == 0:
        acc = np.mean(np.argmax(pred_exp, axis=1) == y_train)
        print(f"Epoch {epoch}, Loss: {loss_exp:.4f}, Acc: {acc:.3f}")